# CBDTP Sentiment — Off-the-Shelf Ensemble + Aspect-Based
### Milestone 2 — sentiment layer (independent of stance)

Scores a **copy** of the anchored-thread corpus with off-the-shelf sentiment models and emits, per item:

* **VALENCE** (overall tone) — a continuous score in **[-1, +1]** from each of **six** models spanning
  **2011→2022**, plus their ensemble.
* **TARGET** (aspect-based) — sentiment **toward** each of a set of debate aspects (the toll's *existence* vs its
  *price*, driving, traffic, the subway, the MTA, NJ commuters, funding, …), via `deberta-v3-large-absa`.

**Two gates** (both default ON): only score items the **vLLM relevance** run marked `about_topic=True`, and run
each aspect's ABSA only on items whose text mentions it. Resumable, GPU-batched; runs in `~/mads-m2-sentiment`.

| key | model | year | family | dim | native → signed |
|---|---|---|---|---|---|
| `afinn`     | AFINN-111 lexicon                                | 2011 | lexicon              | valence | `tanh(sum/scale)` |
| `textblob`  | TextBlob PatternAnalyzer                         | 2013 | lexicon              | valence | `polarity` |
| `vader`     | VADER                                            | 2014 | social lexicon       | valence | `compound` |
| `sst2`      | distilbert-…-sst-2-english                       | 2019 | transformer (binary) | valence | `2·p_pos − 1` |
| `bertweet`  | finiteautomata/bertweet-base-sentiment-analysis  | 2020 | social transformer   | valence | `p_pos − p_neg` |
| `twroberta` | cardiffnlp/twitter-roberta-base-sentiment-latest | 2022 | social transformer   | valence | `p_pos − p_neg` |
| `absa`      | yangheng/deberta-v3-large-absa-v1.1              | 2022 | DeBERTa-v3 (ABSA)    | **target** | `p_pos − p_neg` per aspect |

The **ensemble** averages only the six **valence** models. ABSA is reported as `absa_<aspect>_*` columns (the
natural sentiment companion to stance for RQ4). *(2026 SOTA Snowflake Arctic-ABSA would be the ideal `absa`
model — drop its repo id into the `absa` slot when its HF weights ship.)*

## 0 · Run mode

1. **Single GPU (default):** `WORKER_MODE=False`. Scores every item on `SINGLE_GPU_INDEX` (default `"1"`).
   With relevance-gating ON the workload is the relevant subset only — one card is plenty.
2. **Both GPUs (work-stealing):** Run the `_dp0`/`_dp1` copies. dp0/main **freezes** the ordered worklist once
   (`worklist_{kind}.parquet`); both workers then pull the next unclaimed **chunk** from a shared,
   `flock`-guarded claim queue, score it, and write `chunk_<i>.parquet` into the **shared** output dir. Whoever
   is free grabs the next chunk, so stopping one card lets the other finish everything; a chunk left mid-flight
   by a killed worker is reclaimed after `RECLAIM_TIMEOUT_S`. **Start only after the vLLM relevance run is
   complete** (the worklist is frozen once). §9 merges the shared shards.

Models always run **sequentially within a kernel** (lexicons→valence transformers→ABSA-per-aspect). Knobs
`SENT_BATCH`, `MAX_LEN`, `CHUNK`, `RELEVANCE_GATE`, `GATE_ABSA`, `ASPECT_SET`, `RECLAIM_TIMEOUT_S` are env-overridable.

## 1 · Configuration

In [ ]:
%%time
import os, json, time, math, shutil, zlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import sys
try: sys.stdout.reconfigure(encoding="utf-8")
except Exception: pass

# ══ GPU MODE — edit these lines ═════════════════════════════════════════════════════════════════════════
WORKER_MODE = False        # True = one of the two-GPU worker copies (_dp0/_dp1)
WORKER_GPU  = "1"          # "0" or "1" — set per worker copy (the builder does this for you)
NUM_WORKERS = 1            # workers sharing the corpus (_dp copies set this to 2)
SINGLE_GPU_INDEX = WORKER_GPU
# ════════════════════════════════════════════════════════════════════════════════════════════════════════
WORKER_ID = int(WORKER_GPU) if WORKER_MODE else 0
GPU_INDEX = WORKER_GPU if WORKER_MODE else SINGLE_GPU_INDEX
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_INDEX        # set BEFORE importing torch -> chosen card = cuda:0
print(f"[gpu] {'WORKER '+WORKER_GPU+f' of {NUM_WORKERS}' if WORKER_MODE else 'single GPU '+GPU_INDEX}"
      f" -> CUDA_VISIBLE_DEVICES={GPU_INDEX}", flush=True)

# --- data root (dir containing relevance_filtering-7B-AWQ/) : env -> Win filtering dir -> WSL -> C:\ ---
_CANDS = [os.environ.get("MADS_DATA_DIR"),
          r"G:\Other computers\My MacBook Pro\Documents\UM MADS\696 - Milestone 2\Project Data - 2 - Filtering",
          str(Path.home() / "mads-data"), r"C:\mads-data"]
DATA_DIR = next((Path(c) for c in _CANDS if c and (Path(c) / "relevance_filtering-7B-AWQ").exists()), None)
assert DATA_DIR is not None, "no data root containing relevance_filtering-7B-AWQ/ (set MADS_DATA_DIR)"
CKPT_DIR = DATA_DIR / "relevance_filtering-7B-AWQ"   # READ-ONLY: thr_*.parquet + vLLM labels (the stance job)
SENT_DIR = DATA_DIR / "sentiment-ensemble"           # OUR isolated namespace
SENT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[data] root={DATA_DIR}\n       source={CKPT_DIR}\n       out={SENT_DIR}", flush=True)

# --- scoring knobs ---
SENT_BATCH = int(os.environ.get("SENT_BATCH", "256"))
MAX_LEN    = int(os.environ.get("MAX_LEN", "256"))
CHUNK      = int(os.environ.get("CHUNK", "5000"))    # items per checkpoint part-file
AFINN_SCALE, DEADBAND = 5.0, 0.05

# --- GATING ---
# Relevance gate: only score items the vLLM stance run marked about_topic=True. The vLLM LABEL namespaces only
# (comment_labels-vllm*, vllm_redo_comment_labels, post_labels-vllm*) — NOT the HF *-shard* dirs.
RELEVANCE_GATE = os.environ.get("RELEVANCE_GATE", "1") == "1"
# Tier-1 = high-precision (unambiguous) lexical anchors. We TRUST these and never let the (unvalidated) vLLM
# relevance call drop them -> all items in Tier-1 threads are scored regardless of about_topic. The relevance
# gate then only filters the broader, noisier Tier-2 anchored threads. ~1.15M comments live in Tier-1 threads.
UNGATE_TIER1 = os.environ.get("UNGATE_TIER1", "1") == "1"
# Final consolidated vLLM label files (carry about_topic = relevance, + post-level matched_tier1 / thread_tier1).
LABELED_FILES = {"comment": "labeled_comments-vllm-v5.parquet", "post": "labeled_posts-vllm-v5.parquet"}
# How "Tier-1" (the ungated, always-scored set) is defined for COMMENTS:
#   "comment" = the comment is itself a high-precision lexical hit (hp)                      ~36.9k
#   "branch"  = hp comments PLUS their ancestors + descendants (the on-topic neighborhood)   ~123.6k   [DEFAULT]
#   "thread"  = every comment in a Tier-1 thread (thread_tier1; loosest)                      ~1.15M
# Posts are always post-level (matched_tier1). hp comment anchors come from comment_anchors.parquet.
COMMENT_TIER1_MODE = os.environ.get("COMMENT_TIER1_MODE", "branch")
GATE_ABSA  = os.environ.get("GATE_ABSA", "1") == "1"   # run each aspect's ABSA only on items that mention it
ASPECT_SET = os.environ.get("ASPECT_SET", "broad")     # "core" (9) | "broad" (~25)  -- DEFAULT broad

JUNK = {"[removed]", "[deleted]", "[deleted by user]", "[ Removed by Reddit ]", "", None}

# --- valence panel (+ the single ABSA backend). `dim`: valence feeds the ensemble; target = ABSA per aspect. ---
MODELS = [
    {"key":"afinn",     "type":"lexicon", "year":2011, "dim":"valence"},
    {"key":"textblob",  "type":"lexicon", "year":2013, "dim":"valence"},
    {"key":"vader",     "type":"lexicon", "year":2014, "dim":"valence"},
    {"key":"sst2",      "type":"hf", "year":2019, "dim":"valence", "hf_id":"distilbert-base-uncased-finetuned-sst-2-english"},
    {"key":"bertweet",  "type":"hf", "year":2020, "dim":"valence", "hf_id":"finiteautomata/bertweet-base-sentiment-analysis"},
    {"key":"twroberta", "type":"hf", "year":2022, "dim":"valence", "hf_id":"cardiffnlp/twitter-roberta-base-sentiment-latest"},
    {"key":"absa",      "type":"hf", "year":2022, "dim":"target",  "hf_id":"yangheng/deberta-v3-large-absa-v1.1"},
]
ENABLED = {m["key"]: True for m in MODELS}
MODELS  = [m for m in MODELS if ENABLED.get(m["key"], True)]
LEX_MODELS   = [m for m in MODELS if m["type"]=="lexicon"]
VAL_HF       = [m for m in MODELS if m["type"]=="hf" and m["dim"]=="valence"]
ABSA_MODEL   = next((m for m in MODELS if m["dim"]=="target"), None)
VALENCE_KEYS = [m["key"] for m in MODELS if m["dim"]=="valence"]    # the ensemble keys
VAL_TRANSF_KEYS = [m["key"] for m in VAL_HF]
print(f"[models] valence={VALENCE_KEYS} | absa={'on' if ABSA_MODEL else 'off'} "
      f"| relevance_gate={RELEVANCE_GATE} | gate_absa={GATE_ABSA} | aspect_set={ASPECT_SET} "
      f"| batch={SENT_BATCH} max_len={MAX_LEN} chunk={CHUNK}", flush=True)

## 1b · Aspects (targets for the ABSA model)

Each aspect = **(key, phrase fed to ABSA, generous gate regex)**. The `phrase` is what the model scores
sentiment *toward*; the `gate` decides which items that aspect's ABSA pass even runs on (when `GATE_ABSA`).

**Note the two distinct toll aspects:** `toll_policy` = sentiment toward the toll's *existence* (whether there
should be a toll at all), vs `toll_price` = sentiment toward the *amount* ($9 / the $1.50 rideshare surcharge /
"too high"/"too low"). Someone can favor the policy but think the price is wrong — these must be separate.
`toll_policy` is in `ABSA_NO_GATE` (scored on **every** item, skipping the lexical gate) because the toll's
existence is the corpus's overarching subject and is constantly referenced by pronoun without being named.

Edit freely. `CORE` (9) is the high-signal set; `BROAD` (~25) adds the rest; pick via `ASPECT_SET` in §1.

In [ ]:
%%time
import re
# (key, ABSA aspect phrase, gate regex [case-insensitive, generous = high recall])
CORE = [
 ("toll_policy",   "congestion pricing",        r"congestion pric|congestion toll|congestion charge|congestion relief|\bcbdtp\b|the toll\b|having a toll|whether.{0,15}toll|toll(ing)? (drivers|cars|people)|charging drivers"),
 ("toll_price",    "the toll price",            r"\$9\b|\$15\b|\$1\.?50|nine dollar|fifteen dollar|toll (price|cost|amount|fee)|how much.{0,10}toll|price of the toll|too (high|expensive|cheap|low)"),
 ("driving",       "driving into Manhattan",    r"\bdriv(e|es|ing|ers?)\b|\bcars?\b|motorist|commut\w* by car"),
 ("traffic",       "traffic congestion",        r"\btraffic\b|gridlock|congest(ion|ed)|bumper.?to.?bumper"),
 ("public_transit","public transit",            r"public transit|public transport\w*|mass transit|\btransit system"),
 ("subway",        "the subway",                r"\bsubway\b|\btrains?\b|\b[1-7] train\b|\b[a-z] train\b|\bmetro\b"),
 ("mta",           "the MTA",                   r"\bmta\b|\bm\.t\.a\.?\b|lieber|metropolitan transportation|transit authority"),
 ("nj_commuters",  "New Jersey commuters",      r"new jersey|\bnj\b|jersey|out.?of.?state|connecticut|\bct\b|across the (river|bridge|tunnel)"),
 ("mta_funding",   "funding for the MTA",       r"\brevenue\b|\bfund(s|ing)?\b|capital (plan|program)|\bbudget\b|pay for|money for|where.{0,15}money go|\$\d+ ?billion"),
]
EXTENDED = [
 ("buses",            "the buses",                  r"\bbus(es|ses)?\b"),
 ("commuter_rail",    "commuter rail",              r"\blirr\b|long island rail|metro.?north|nj ?transit|\bpath\b|commuter (rail|train)"),
 ("parking",          "parking",                    r"\bpark(ing|ed)?\b|\bgarage\b|\bmeter\b"),
 ("bike_lanes",       "bike lanes",                 r"bike ?lane|cycl(e|ing|ist)|\bbikes?\b|\bebikes?\b|citi ?bike"),
 ("fares_omny",       "transit fares",              r"\bomny\b|\bfare[s]?\b|metrocard|\bswipe\b|fare (hike|evasion)"),
 ("transit_safety",   "safety on public transit",   r"\bcrime\b|\bsafe(ty)?\b|unsafe|assault|stabb|shov(e|ed)|push(ed)? onto|attack|mugg"),
 ("transit_cleanliness","cleanliness of the subway", r"\bdirty\b|\bfilth|\bsmell|\bclean\b|\brats?\b|\bpiss|homeless|urine"),
 ("middle_class",     "the middle class",           r"middle class|working class|regular (people|folks)|average (person|new yorker)"),
 ("low_income_equity","low-income people",          r"low.?income|\bpoor\b|\bequity\b|regressive|\bfair(ness)?\b|disproportion|burden"),
 ("taxes",            "taxes",                      r"\btax(es|payer|ed|ing)?\b|another tax|double tax|cash grab|money grab"),
 ("small_business",   "small businesses",           r"small business|\bshop(s|keeper)?\b|store owner|restaurant owner|\bvendors?\b"),
 ("trump_federal",    "the federal government",     r"\btrump\b|federal|white house|\bdot\b|\bduffy\b|\bfhwa\b|washington"),
 ("hochul",           "Governor Hochul",            r"\bhochul\b|the governor\b|\bkathy\b"),
 ("lawsuits",         "lawsuits against the toll",  r"lawsuit|\bsue[ds]?\b|\bcourt\b|\bjudge\b|legal|injunction|litigat"),
 ("outer_boroughs",   "the outer boroughs",         r"outer borough|\bqueens\b|\bbronx\b|\bbrooklyn\b|staten island"),
 ("manhattan_cbd",    "driving in Manhattan",       r"\bmanhattan\b|below 60th|60th st|central business|\bcbd\b|downtown|midtown"),
]
ASPECTS = CORE if ASPECT_SET == "core" else (CORE + EXTENDED)
ASPECTS = [(k, phrase, re.compile(g, re.I)) for (k, phrase, g) in ASPECTS]   # precompile gates
ABSA_KEYS = [k for (k, _, _) in ASPECTS]
# Aspects scored on EVERY (scorable) item, skipping the lexical gate. The toll's existence is the overarching
# subject of the relevant corpus and is often referred to by pronoun ("it"/"this") without naming it, so gating
# it on keywords would drop real signal -> keep toll_policy ungated.
ABSA_NO_GATE = {"toll_policy"}
print(f"[aspects] {ASPECT_SET}: {len(ASPECTS)} -> {ABSA_KEYS} | ungated: {sorted(ABSA_NO_GATE & set(ABSA_KEYS))}", flush=True)

### Checkpoint, resume & the work-stealing claim queue

Immutable parquet shards in a **shared** `{kind}_sentiment/` dir.
* **Single-GPU:** append `part_*.parquet`; resume by subtracting already-written ids.
* **Two-GPU workers (`_dp0`/`_dp1`):** dp0/main freezes the ordered worklist (`worklist_{kind}.parquet`), then
  each worker repeatedly **claims** the next unclaimed **chunk index** (guarded by `fcntl.flock`), scores it, and
  writes `chunk_<i>.parquet`. Whoever is free grabs the next chunk → real work-stealing. A chunk claimed but
  unfinished past `RECLAIM_TIMEOUT_S` is reclaimable (covers a killed worker). Resume = the chunk files that
  already exist. Needs a STABLE worklist → start the workers only after the vLLM relevance run is complete.

In [ ]:
%%time
import contextlib
RECLAIM_TIMEOUT_S = int(os.environ.get("RECLAIM_TIMEOUT_S", "1800"))   # a chunk claimed but unfinished this long -> reclaimable
WID = f"gpu{WORKER_GPU}"
def sp(name): return SENT_DIR / name
def save_json_atomic(obj, name):
    tmp = sp(name + ".tmp"); tmp.write_text(json.dumps(obj)); tmp.replace(sp(name))

def parts_dir(kind): return f"{kind}_sentiment"        # SHARED across workers (the claim queue prevents collisions)

# ---- single-GPU shards (id-resume) ----
def append_parts(df, subdir):
    d = sp(subdir); d.mkdir(parents=True, exist_ok=True)
    k = len(list(d.glob("part_*.parquet")))
    df.to_parquet(d / f"part_{k:05d}.parquet")
def done_ids(subdir):
    d = sp(subdir); ids = set()
    if d.exists():
        for f in d.glob("*.parquet"):
            try: ids |= set(pd.read_parquet(f, columns=["id"])["id"])
            except Exception: pass
    return ids
def load_parts_glob(base):                  # merge every shard (part_* and chunk_*) under dirs named like `base`
    dfs = []
    for d in sorted(SENT_DIR.glob(f"{base}*")):
        if d.is_dir():
            fs = sorted(f for f in d.glob("*.parquet") if not f.name.endswith(".tmp"))
            if fs: dfs.append(pd.concat((pd.read_parquet(f) for f in fs), ignore_index=True))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# ---- frozen worklist (worker mode): stager builds it once, others wait + read ----
def ensure_worklist(kind, build_fn):
    wpath = sp(f"worklist_{kind}.parquet")
    if not wpath.exists() and (not WORKER_MODE or WORKER_GPU == "0"):     # dp0 / main is the stager
        wl = build_fn(kind)
        tmp = wpath.with_suffix(".parquet.tmp"); wl.to_parquet(tmp); tmp.replace(wpath)
        print(f"[worklist] froze {kind}: {len(wl):,} items -> {wpath.name}", flush=True); return wl
    waited = 0
    while not wpath.exists():
        if waited == 0: print(f"[worklist] waiting for dp0/main to freeze {wpath.name} ...", flush=True)
        time.sleep(3); waited += 3
        if waited > 1800: raise SystemExit(f"[worklist] timed out waiting for {wpath.name}; run dp0/main to freeze it.")
    return pd.read_parquet(wpath)

# ---- work-stealing claim queue (worker mode) ----
@contextlib.contextmanager
def _claim_lock(kind):
    # fcntl.flock is Linux-only, auto-releases if the process dies; held ONLY for the ms-long claim bookkeeping,
    # never during scoring. Absent on Windows -> no-op (single-process is safe anyway).
    try:
        import fcntl
        f = open(sp(f"claims_{kind}.lock"), "w")
        try: fcntl.flock(f, fcntl.LOCK_EX); yield
        finally: fcntl.flock(f, fcntl.LOCK_UN); f.close()
    except ImportError:
        yield
def _load_claims(kind):
    p = sp(f"claims_{kind}.json"); return json.loads(p.read_text()) if p.exists() else {}
def claim_next(kind, n_chunks, pdir):
    with _claim_lock(kind):
        claims = _load_claims(kind); now = time.time()
        for ci in range(n_chunks):
            if (sp(pdir) / f"chunk_{ci:06d}.parquet").exists(): continue              # already finished
            c = claims.get(str(ci))
            if c and c.get("status") == "claimed" and now - c.get("ts", 0) < RECLAIM_TIMEOUT_S: continue
            claims[str(ci)] = {"worker": WID, "status": "claimed", "ts": now}
            save_json_atomic(claims, f"claims_{kind}.json"); return ci
    return None
def mark_done(kind, ci):
    with _claim_lock(kind):
        claims = _load_claims(kind); claims[str(ci)] = {"worker": WID, "status": "done", "ts": time.time()}
        save_json_atomic(claims, f"claims_{kind}.json")
def write_chunk(df, subdir, ci):            # deterministic name -> two workers never collide
    d = sp(subdir); d.mkdir(parents=True, exist_ok=True)
    p = d / f"chunk_{ci:06d}.parquet"; tmp = d / f"chunk_{ci:06d}.parquet.tmp"
    df.to_parquet(tmp); tmp.replace(p)
print("checkpoint + claim-queue helpers ready | parts:", parts_dir("comment"), flush=True)

## 2 · Make the working COPY

Score on a copy so the text reads never touch the stance run's files. Static sources; skipped if already copied.

In [ ]:
%%time
SRC = {"post": CKPT_DIR / "thr_posts.parquet", "comment": CKPT_DIR / "thr_comments.parquet"}
CPY = {"post": SENT_DIR / "src_posts.parquet",     "comment": SENT_DIR / "src_comments.parquet"}
def _staged(src, dst): return dst.exists() and dst.stat().st_size == src.stat().st_size
for kind in ("post", "comment"):
    src, dst = SRC[kind], CPY[kind]
    assert src.exists(), f"source missing: {src}"
    if _staged(src, dst):
        print(f"[copy] {kind}: already staged ({dst.stat().st_size/1e6:.0f} MB)", flush=True)
    elif WORKER_MODE and WORKER_GPU != "0":
        # dp1 (and any non-0 worker) waits for dp0/main to stage the copy, then proceeds (no race, no half-file).
        waited = 0
        while not _staged(src, dst):
            if waited == 0: print(f"[copy] {kind}: waiting for dp0/main to stage the copy ...", flush=True)
            time.sleep(3); waited += 3
            if waited > 1200: raise SystemExit(f"[copy] timed out waiting for {dst.name}; run dp0 or the main notebook to stage it.")
        print(f"[copy] {kind}: ready", flush=True)
    else:
        # main or dp0 stages it; atomic temp->rename so a watcher never sees a half-written file.
        print(f"[copy] {kind}: copying {src.name} ({src.stat().st_size/1e6:.0f} MB) ...", flush=True)
        tmp = dst.with_suffix(dst.suffix + ".tmp"); shutil.copyfile(src, tmp); tmp.replace(dst)
        print(f"[copy] {kind}: done", flush=True)

## 3 · Relevance + Tier labels (from the consolidated vLLM label files)

`is_vllm_relevant = about_topic` (from the labeled files). The **Tier-1 ungate set** (`tier==1`) differs by kind:

* **posts → `matched_tier1`** (post-level): the post is itself a high-precision lexical hit (~3,588). A post
  that's Tier-1 only because of a high-precision *comment* is **not** force-included — vLLM relevance decides.
* **comments → `COMMENT_TIER1_MODE`** (default **`"branch"`**): the high-precision (`hp`) comments **plus their
  ancestors + descendants** — the on-topic neighborhood around each anchor (~123.6k). The branch set rescues
  context-dependent replies near anchors (where the LLM is most likely to false-negative) while excluding distant
  off-topic side-branches. Alternatives: `"comment"` (hp only) / `"thread"` (whole Tier-1 thread). `hp` anchors
  come from `comment_anchors.parquet`.

We also carry **`is_hp_anchor`** (the item is itself a high-precision hit) for the finest downstream cut.

**Scope** (when `RELEVANCE_GATE`): `tier==1` (if `UNGATE_TIER1`) **OR** `is_vllm_relevant`. With `"branch"`:
~438k comments (123.6k neighborhood ∪ 382.6k relevant), 4,075 posts. Tier-1 is never gated (unvalidated vLLM
shouldn't drop unambiguous anchors); the gate only filters **Tier-2**.

In [ ]:
%%time
def _find_labeled(fn):
    for d in [DATA_DIR, CKPT_DIR, SENT_DIR,
              Path(r"G:\Other computers\My MacBook Pro\Documents\UM MADS\696 - Milestone 2\Project Data - 4 - Sentiment")]:
        if (d / fn).exists(): return d / fn
    return None

LAB = {}
for kind, fn in LABELED_FILES.items():
    p = _find_labeled(fn)
    assert p is not None, (f"labeled file {fn} not found — copy it next to the data (e.g. ~/mads-data/) "
                           f"or set MADS_DATA_DIR to its folder")
    cols = ["id", "about_topic"] + (["matched_tier1"] if kind == "post" else [])
    df = pd.read_parquet(p, columns=cols); df["id"] = df["id"].astype(str)
    LAB[kind] = df.set_index("id")
    print(f"[labels] {kind}: {p.name} rows={len(df):,} | relevant(about_topic)={int(df['about_topic'].sum()):,}", flush=True)
POST_TIER1_IDS = set(LAB["post"].index[LAB["post"]["matched_tier1"] == True])      # post-level Tier-1 (~3,588)

# comment-level high-precision anchors (hp) -> basis for the comment Tier-1 set
_ca = pd.read_parquet(CKPT_DIR / "comment_anchors.parquet", columns=["id", "link_id", "hp"])
_ca["id"] = _ca["id"].astype(str); _ca["link_id"] = _ca["link_id"].astype(str)
HP_COMMENT_IDS     = set(_ca.loc[_ca["hp"] == True, "id"])
HP_COMMENT_THREADS = set(_ca.loc[_ca["hp"] == True, "link_id"])
print(f"[tier1] post-level Tier-1={len(POST_TIER1_IDS):,} | hp comments={len(HP_COMMENT_IDS):,} "
      f"in {len(HP_COMMENT_THREADS):,} threads | COMMENT_TIER1_MODE={COMMENT_TIER1_MODE}", flush=True)

_CT1 = {"ids": None}
def comment_tier1_ids():
    """The comment ids treated as Tier-1 (ungated), per COMMENT_TIER1_MODE. Built once (stager) and cached."""
    if _CT1["ids"] is not None: return _CT1["ids"]
    mode = COMMENT_TIER1_MODE
    if mode == "comment":
        ids = set(HP_COMMENT_IDS)
    elif mode == "thread":
        lab = pd.read_parquet(_find_labeled(LABELED_FILES["comment"]), columns=["id", "thread_tier1"])
        ids = set(lab.loc[lab["thread_tier1"] == True, "id"].astype(str))
    else:  # "branch" = hp comments + ancestors + descendants
        from collections import defaultdict
        tc = pd.read_parquet(CPY["comment"], columns=["id", "parent_id", "link_id"])
        tc["id"] = tc["id"].astype(str); tc["parent_id"] = tc["parent_id"].astype(str); tc["link_id"] = tc["link_id"].astype(str)
        sub = tc[tc["link_id"].isin(HP_COMMENT_THREADS)]
        pid = {}; ch = defaultdict(list)
        for i, p in zip(sub["id"], sub["parent_id"]):
            par = p[3:] if p.startswith("t1_") else None      # t1_<cid> = reply to a comment; t3_<post> = top-level
            pid[i] = par
            if par is not None: ch[par].append(i)
        ids = set(HP_COMMENT_IDS)
        for a in HP_COMMENT_IDS:                              # ancestors: the context the anchor is replying within
            cur = pid.get(a)
            while cur is not None and cur in pid and cur not in ids:   # `cur in pid` skips phantom (deleted) parents
                ids.add(cur); cur = pid.get(cur)
        st = list(HP_COMMENT_IDS)                             # descendants: replies under the anchor
        while st:
            x = st.pop()
            for c in ch.get(x, ()):
                if c not in ids: ids.add(c); st.append(c)
    _CT1["ids"] = ids
    print(f"[tier1] comment Tier-1 set ({mode}): {len(ids):,}", flush=True)
    return ids
print(f"[gate] RELEVANCE_GATE={RELEVANCE_GATE} UNGATE_TIER1={UNGATE_TIER1} | scope = (Tier-1 if ungated) OR relevant", flush=True)

## 4 · Load the copy, build text, apply the relevance gate

posts: `title`+`selftext`; comments: `body`. Junk/empty kept but flagged `scorable=False` (never sent to a
model). With the relevance gate, only `about_topic` items are retained.

In [ ]:
%%time
def clean(s):
    s = "" if s is None else str(s)
    return "" if s.strip() in {"[removed]","[deleted]","[deleted by user]","[ Removed by Reddit ]"} else s

def load_kind(kind):
    if kind == "post":
        df = pd.read_parquet(CPY["post"], columns=["id","name","title","selftext","subreddit"])
        df["id"] = df["id"].astype(str)
        text = (df["title"].map(clean).fillna("") + "\n\n" + df["selftext"].map(clean).fillna("")).str.strip()
        out = pd.DataFrame({"id": df["id"], "kind": "post", "link_id": df["name"].astype(str),
                            "subreddit": df["subreddit"].astype(str), "created_utc": pd.NA, "text": text})
    else:
        df = pd.read_parquet(CPY["comment"], columns=["id","link_id","created_utc","body"])
        df["id"] = df["id"].astype(str)
        text = df["body"].map(clean).fillna("").str.strip()
        out = pd.DataFrame({"id": df["id"], "kind": "comment", "link_id": df["link_id"].astype(str),
                            "subreddit": pd.NA, "created_utc": df["created_utc"], "text": text})
    # labels carried into every output row (independent of gating)
    out["is_vllm_relevant"] = out["id"].map(LAB[kind]["about_topic"]).fillna(False).astype(bool)
    if kind == "post":
        out["is_hp_anchor"] = out["id"].isin(POST_TIER1_IDS)      # post is itself a high-precision hit
        t1 = out["is_hp_anchor"].to_numpy()
    else:
        out["is_hp_anchor"] = out["id"].isin(HP_COMMENT_IDS)      # comment is itself a high-precision hit
        t1 = out["id"].isin(comment_tier1_ids()).to_numpy()       # Tier-1 set per COMMENT_TIER1_MODE
    out["tier"] = np.where(t1, 1, 2).astype("int8")               # 1 = Tier-1 (ungated), 2 = relied on vLLM relevance
    if RELEVANCE_GATE:
        keep = out["is_vllm_relevant"].to_numpy()                 # Tier-2: trust the vLLM relevance call
        if UNGATE_TIER1:
            keep = keep | (out["tier"].to_numpy() == 1)           # Tier-1: always score
        out = out[keep].reset_index(drop=True)
    out["text_len"] = out["text"].str.len().astype("int32")
    out["scorable"] = out["text_len"] > 0
    return out
print("loader ready")

## 5 · Model backends (signed-score mapping)

Valence → `<k>_signed ∈ [-1,1]` (+ native columns). ABSA → per aspect `absa_<key>_signed/label/conf` from the
`(text, aspect)` pair, scored only on gated items (NaN elsewhere). Discrete labels use a `±DEADBAND` neutral zone.

In [ ]:
%%time
from afinn import Afinn
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
_afinn, _vader = Afinn(), SentimentIntensityAnalyzer()
def _disc(sig): return "pos" if sig > DEADBAND else ("neg" if sig < -DEADBAND else "neu")

def score_lexicons(texts):
    cols = {}
    raw = np.array([_afinn.score(t) for t in texts], dtype="float32"); sig = np.tanh(raw/AFINN_SCALE).astype("float32")
    cols["afinn_raw"]=raw; cols["afinn_signed"]=sig; cols["afinn_label"]=[_disc(s) for s in sig]
    pol=[TextBlob(t).sentiment.polarity for t in texts]; sig=np.array(pol,dtype="float32")
    cols["textblob_signed"]=sig; cols["textblob_label"]=[_disc(s) for s in sig]
    comp=[]; pos=[]; neu=[]; neg=[]
    for t in texts:
        d=_vader.polarity_scores(t); comp.append(d["compound"]); pos.append(d["pos"]); neu.append(d["neu"]); neg.append(d["neg"])
    sig=np.array(comp,dtype="float32")
    cols["vader_signed"]=sig; cols["vader_pos"]=np.array(pos,dtype="float32")
    cols["vader_neu"]=np.array(neu,dtype="float32"); cols["vader_neg"]=np.array(neg,dtype="float32")
    cols["vader_label"]=[_disc(s) for s in sig]
    return cols

_PIPES = {}
_PIPE_MAXLEN = {}     # per-model truncation cap (some models — e.g. BERTweet — have a short position table)
def _norm3(label):
    l=str(label).lower(); return "neg" if l.startswith("neg") else ("pos" if l.startswith("pos") else "neu")
def ensure_pipes():
    if _PIPES: return
    import torch
    from transformers import pipeline
    dev = 0 if torch.cuda.is_available() else -1
    dt = torch.float16 if dev==0 else None
    print(f"[hf] loading {len(VAL_HF)+(1 if ABSA_MODEL else 0)} transformer(s) on {'cuda:0' if dev==0 else 'cpu'} ...", flush=True)
    for m in VAL_HF + ([ABSA_MODEL] if ABSA_MODEL else []):
        pipe = pipeline("sentiment-analysis", model=m["hf_id"], tokenizer=m["hf_id"],
                        top_k=None, device=dev, torch_dtype=dt, function_to_apply="softmax")
        _PIPES[m["key"]] = pipe
        # CAP truncation to the model's positional limit. RoBERTa-family (incl. BERTweet) reserve 2 positions for
        # the padding offset; BERTweet-base has max_position_embeddings=130 -> only ~128 usable. Forcing a longer
        # max_length overruns the position-embedding table -> CUDA device-side assert. So cap per model.
        mp = int(getattr(pipe.model.config, "max_position_embeddings", 0) or 0)
        _PIPE_MAXLEN[m["key"]] = max(16, min(MAX_LEN, mp - 2)) if mp else MAX_LEN
        print(f"[hf]   {m['key']:9s} <- {m['hf_id']} | max_len={_PIPE_MAXLEN[m['key']]}", flush=True)

def _probs(res):                              # pipeline (top_k=None) rows -> (pos,neu,neg,label,conf) arrays
    n=len(res); pos=np.zeros(n,"float32"); neu=np.zeros(n,"float32"); neg=np.zeros(n,"float32")
    lab=[]; conf=np.zeros(n,"float32")
    for i,scores in enumerate(res):
        d={"neg":0.0,"neu":0.0,"pos":0.0}
        for s in scores: d[_norm3(s["label"])] += float(s["score"])
        pos[i],neu[i],neg[i]=d["pos"],d["neu"],d["neg"]; top=max(d,key=d.get); lab.append(top); conf[i]=d[top]
    return pos,neu,neg,lab,conf

def score_valence_transformers(texts):
    cols={}
    for m in VAL_HF:
        k=m["key"]
        res=_PIPES[k](list(texts), batch_size=SENT_BATCH, truncation=True, max_length=_PIPE_MAXLEN[k], padding=True)
        pos,neu,neg,lab,conf=_probs(res)
        cols[f"{k}_pos"]=pos; cols[f"{k}_neu"]=neu; cols[f"{k}_neg"]=neg
        cols[f"{k}_signed"]=(pos-neg).astype("float32"); cols[f"{k}_label"]=lab; cols[f"{k}_conf"]=conf
    return cols

def absa_score(texts, aspect_phrase):         # one aspect over a list of texts -> (signed,label,conf,pos,neu,neg)
    inp=[{"text":t, "text_pair":aspect_phrase} for t in texts]
    res=_PIPES["absa"](inp, batch_size=SENT_BATCH, truncation=True, max_length=_PIPE_MAXLEN["absa"], padding=True)
    pos,neu,neg,lab,conf=_probs(res)
    return (pos-neg).astype("float32"), lab, conf, pos, neu, neg
print("backends ready")

## 6 · Per-chunk scoring

In [ ]:
%%time
ENS_SIGNED = [f"{k}_signed" for k in VALENCE_KEYS]
ENS_TRANSF = [f"{k}_signed" for k in VAL_TRANSF_KEYS]
MODEL_VERSIONS = json.dumps({**{m["key"]: m.get("hf_id", m["type"]) for m in MODELS},
                             "aspects": {k: ph for (k, ph, _) in ASPECTS}})

def _scatter(out, n, pos_idx, col, vals):
    arr=np.asarray(vals)
    if arr.dtype.kind in "fiu":
        full=np.full(n, np.nan, "float32"); full[pos_idx]=arr.astype("float32")
    else:
        full=np.array(["none"]*n, dtype=object); full[pos_idx]=arr
    out[col]=full

def score_chunk(block_df):
    base=block_df[["id","kind","link_id","subreddit","created_utc","tier","is_vllm_relevant","is_hp_anchor","text_len","scorable"]].reset_index(drop=True)
    n=len(base); texts=block_df["text"].to_numpy()
    sc=base["scorable"].to_numpy(); pos_idx=np.where(sc)[0]
    out=base.copy()
    # --- valence (lexicons + transformers) on scorable items ---
    if len(pos_idx):
        todo=texts[pos_idx].tolist()
        for col,vals in {**score_lexicons(todo), **score_valence_transformers(todo)}.items():
            _scatter(out, n, pos_idx, col, vals)
    else:
        for k in VALENCE_KEYS:
            out[f"{k}_signed"]=np.full(n, np.nan, "float32")
    M=out[[c for c in ENS_SIGNED if c in out]].to_numpy("float32")
    with np.errstate(invalid="ignore"):
        out["ensemble_signed"]=np.nanmean(M,axis=1).astype("float32")
        out["ensemble_signed_std"]=np.nanstd(M,axis=1).astype("float32")
    tc=[c for c in ENS_TRANSF if c in out]
    if tc: out["ensemble_signed_transformers"]=np.nanmean(out[tc].to_numpy("float32"),axis=1).astype("float32")
    out["ensemble_label"]=[(_disc(s) if np.isfinite(s) else "none") for s in out["ensemble_signed"]]
    # --- ABSA per aspect, on items that mention the aspect (or all scorable if GATE_ABSA off) ---
    if ABSA_MODEL:
        low=np.array([t.lower() for t in texts], dtype=object)
        for (k, phrase, gate) in ASPECTS:
            if GATE_ABSA and k not in ABSA_NO_GATE:
                hit=np.array([bool(gate.search(low[i])) for i in range(n)]) & sc
            else:
                hit=sc.copy()           # ungated aspect (or GATE_ABSA off) -> score all scorable items
            ai=np.where(hit)[0]
            # build writable numpy arrays, fill at the gated rows, then assign once (never mutate a df-backed
            # to_numpy() view -> that's read-only under pandas Copy-on-Write).
            cols={suf: np.full(n, np.nan, "float32") for suf in ("signed","conf","pos","neu","neg")}
            lab_full=np.array(["none"]*n, dtype=object)
            if len(ai):
                sig,lab,conf,p,nu,ng=absa_score(texts[ai].tolist(), phrase)
                cols["signed"][ai]=sig; cols["conf"][ai]=conf; cols["pos"][ai]=p; cols["neu"][ai]=nu; cols["neg"][ai]=ng
                lab_full[ai]=lab
            for suf,arr in cols.items(): out[f"absa_{k}_{suf}"]=arr
            out[f"absa_{k}_label"]=lab_full
    out["model_versions"]=MODEL_VERSIONS; out["scored_at"]=time.strftime("%Y-%m-%d %H:%M:%S")
    return out
print("chunk scorer ready")

## 7 · Runner — single-GPU (id-resume) or two-GPU work-stealing

In [ ]:
%%time
def run_kind(kind):
    pdir = parts_dir(kind)
    if WORKER_MODE:
        wl = ensure_worklist(kind, load_kind)                 # dp0/main freezes; this worker waits + reads
        n = len(wl); n_chunks = math.ceil(n / CHUNK) if n else 0
        done0 = len(list(sp(pdir).glob("chunk_*.parquet"))) if sp(pdir).exists() else 0
        print(f"[{kind}] worklist={n:,} | chunks={n_chunks:,} | already done={done0:,} | worker={WID} (work-stealing)", flush=True)
        if not n_chunks: return
        ensure_pipes()
        t0 = time.time(); did = 0
        while True:
            ci = claim_next(kind, n_chunks, pdir)             # next chunk no one else is actively doing
            if ci is None: break                              # nothing left to steal
            block = wl.iloc[ci*CHUNK:(ci+1)*CHUNK]
            write_chunk(score_chunk(block), pdir, ci); mark_done(kind, ci)
            did += 1; el = time.time() - t0
            tot = len(list(sp(pdir).glob("chunk_*.parquet")))
            print(f"[ckpt] {WID} chunk {ci} (+{len(block):,}) | this worker {did} chunks @ {did*CHUNK/max(el,1e-6):,.0f} rows/s "
                  f"| {tot}/{n_chunks} chunks total", flush=True)
        print(f"[{kind}] {WID}: no unclaimed chunks left — done", flush=True)
    else:
        df = load_kind(kind)
        done = done_ids(pdir)
        todo = df[~df["id"].isin(done)].reset_index(drop=True)
        print(f"[{kind}] candidates={len(df):,} | already scored={len(done):,} | to score now={len(todo):,} (chunk={CHUNK:,})", flush=True)
        if not len(todo): return
        ensure_pipes()
        t0 = time.time(); prev = 0.0
        for gs in range(0, len(todo), CHUNK):
            block = todo.iloc[gs:gs+CHUNK]
            append_parts(score_chunk(block), pdir)
            el = time.time()-t0; nr = len(block); batch = nr/max(el-prev,1e-6); prev = el; cum = (gs+nr)/max(el,1e-6)
            print(f"[ckpt] +{nr:,} -> {gs+nr:,}/{len(todo):,} | {el:.0f}s | {cum:,.0f} rows/s cum | {batch:,.0f} rows/s batch", flush=True)
print("runner ready")

## 7b · ABSA throughput benchmark (optional — run before the full job)

Times the ABSA model on a sample so the per-aspect cost is a **measured** number on your GPU, not an estimate.

In [ ]:
%%time
def absa_benchmark(n=2000, aspect="congestion pricing"):
    d=load_kind("comment"); d=d[d["scorable"]].head(n)
    if not len(d): print("[bench] no scorable comments (relevance gate may be empty yet)"); return
    ensure_pipes()
    t=time.time(); absa_score(d["text"].tolist(), aspect); dt=time.time()-t
    r=len(d)/dt
    print(f"[bench] ABSA {len(d):,} items in {dt:.1f}s = {r:,.0f} items/s/aspect", flush=True)
    print(f"[bench] -> per aspect over N items: N/{r:,.0f} s. With GATE_ABSA, N = items mentioning that aspect.", flush=True)
# absa_benchmark()   # uncomment to run

## 8a · Score posts

In [ ]:
%%time
run_kind("post")

## 8b · Score comments (the long pass — resumable; re-run to extend as the stance job labels more)

In [ ]:
%%time
run_kind("comment")

## 9 · Merge, manifest & summary

In [ ]:
%%time
def finalize(kind):
    df=load_parts_glob(f"{kind}_sentiment").drop_duplicates("id")
    if not len(df): print(f"[final] {kind}: nothing scored yet"); return None
    # Re-stamp relevance from the CURRENT labeled file so the merged column is consistent even when this run only
    # APPENDED a delta on top of earlier shards (e.g. the v5 redo relabels some already-scored items as relevant).
    flipped=int((df["id"].map(LAB[kind]["about_topic"]).fillna(df["is_vllm_relevant"]).astype(bool) != df["is_vllm_relevant"]).sum())
    df["is_vllm_relevant"]=df["id"].map(LAB[kind]["about_topic"]).fillna(df["is_vllm_relevant"]).astype(bool)
    df.to_parquet(sp(f"{kind}_sentiment.parquet"))
    print(f"[final] {kind}: {len(df):,} rows (re-stamped is_vllm_relevant on {flipped:,})", flush=True)
    return df

man={"generated_at": time.strftime("%Y-%m-%d %H:%M:%S"), "models": MODEL_VERSIONS,
     "relevance_gate": RELEVANCE_GATE, "gate_absa": GATE_ABSA, "aspect_set": ASPECT_SET,
     "max_len": MAX_LEN, "kinds": {}}
for kind in ("post","comment"):
    df=finalize(kind)
    if df is None: continue
    sc=df[df["scorable"]]
    val_means={k: float(np.nanmean(df[f"{k}_signed"])) for k in VALENCE_KEYS if f"{k}_signed" in df}
    absa_cov={k: int(df[f"absa_{k}_signed"].notna().sum()) for k in ABSA_KEYS if f"absa_{k}_signed" in df}
    absa_mean={k: (float(np.nanmean(df[f"absa_{k}_signed"])) if df[f"absa_{k}_signed"].notna().any() else None)
               for k in ABSA_KEYS if f"absa_{k}_signed" in df}
    man["kinds"][kind]={"n":int(len(df)), "n_scorable":int(len(sc)),
                        "n_tier1": int((df["tier"]==1).sum()), "n_tier2": int((df["tier"]==2).sum()),
                        "n_vllm_relevant": int(df["is_vllm_relevant"].sum()), "n_hp_anchor": int(df["is_hp_anchor"].sum()),
                        "ensemble_label_counts": df["ensemble_label"].value_counts().to_dict(),
                        "valence_mean_signed": val_means, "absa_coverage": absa_cov, "absa_mean_signed": absa_mean}
    print(f"\n=== {kind.upper()} === n={len(df):,} scorable={len(sc):,} "
          f"| tier1={int((df['tier']==1).sum()):,} tier2={int((df['tier']==2).sum()):,} "
          f"vllm_relevant={int(df['is_vllm_relevant'].sum()):,} hp_anchor={int(df['is_hp_anchor'].sum()):,}")
    print("valence ensemble label:", df["ensemble_label"].value_counts().to_dict())
    print("valence per-model mean (time order):")
    for m in MODELS:
        k = m["key"]
        if m["dim"]=="valence" and f"{k}_signed" in df:
            print(f"   {m['year']}  {k:10s} {np.nanmean(df[f'{k}_signed']):+.3f}")
    print(f"   valence-ensemble mean: {np.nanmean(df['ensemble_signed']):+.3f}")
    print("ABSA per aspect  [coverage | mean signed toward aspect]:")
    for k in ABSA_KEYS:
        if f"absa_{k}_signed" in df:
            cov=absa_cov[k]; mn=absa_mean[k]
            print(f"   {k:18s} {cov:>8,} items | {('%+.3f'%mn) if mn is not None else '   n/a'}")
save_json_atomic(man, "manifest-sentiment.json")
print("\n[manifest] wrote manifest-sentiment.json", flush=True)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>